# 7. Symbolic optics

LUMOS can carry fields as **closed-form expressions** (SymPy) instead of sampled arrays. This notebook shows:

- symbolic fields, their parameters and their LaTeX rendering;
- Gaussian beams propagated **exactly** through lenses and free space (ABCD law of the complex parameter q);
- tabulated transforms: a circular pupil focused by a lens gives the Airy pattern in closed form;
- the automatic, logged conversion to numbers before elements without symbolic transfer (detectors) or
  without closed form (pupils with spiders);
- the way back: from sampled fields to expressions (Zernike projection, Gaussian fit).

In [ ]:
# On Google Colab only: install LUMOS and fetch the presets used by the examples.
import sys

if "google.colab" in sys.modules:
    %pip install -q "lumos-maxima[catalogs] @ git+https://github.com/VForiel/LUMOS"
    !git clone -q --depth 1 https://github.com/VForiel/LUMOS /content/LUMOS
    !cp -r /content/LUMOS/examples/presets /content/LUMOS/examples/catalog_cache .

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from astropy import units as u

import lumos.io
from lumos import Context
from lumos.backend import to_numpy
from lumos.elements.apertures import CircularAperture
from lumos.elements.detectors import PowerMeter
from lumos.elements.optics import FreeSpace, ThinLens
from lumos.elements.sources import GaussianLaser
from lumos.fields import Grid, Wavefront
from lumos.physics.zernike import zernike_basis
from lumos.symbolic import SymbolicField, SymbolicSource, gaussian_beam, reconstruct, x, y
from lumos.symbolic.evaluate import to_wavefront

## A symbolic field

A `SymbolicField` holds an expression of the canonical variables `x, y` (m) and of named parameters given
as quantities. Jupyter renders it in LaTeX.

In [ ]:
w = sp.Symbol("w", positive=True)
field = SymbolicField(sp.exp(-(x**2 + y**2) / w**2), parameters={w: 1 * u.mm}, wavelength=633 * u.nm)
field

## A Gaussian beam focused by a lens

The preset `presets/symbolic_focus.lumos` focuses a Nd:YAG beam (w₀ = 2 mm) with a lens of 500 mm and
propagates it to its new waist. Each element applies its ABCD matrix to the complex parameter q, so the field
stays a closed form; the power meter at the end needs numbers, and the journal records the conversion.

In [ ]:
ctx = lumos.io.load("presets/symbolic_focus.lumos")
result = ctx.run(ctx.meta["outputs"], archive="symbolic-optics")
focused = result["gap.out"]
print(type(focused).__name__, "- power read by the meter:", result["meter.out"].data)
[entry.message for entry in ctx.journal.entries("log") if "symbolic" in entry.message]

The new waist predicted by Gaussian optics (Self 1983) is
$w_0' = \frac{\lambda f}{\pi w_0} / \sqrt{1 + (f/z_R)^2}$. The closed form carried by the field gives it
directly:

In [ ]:
lam, w0, f = 1.064e-6, 2e-3, 0.5
z_r = math.pi * w0**2 / lam
expected = lam * f / (math.pi * w0) / math.sqrt(1 + (f / z_r) ** 2)
print(f"closed form: {focused.form.width() * 1e6:.4f} µm, Gaussian optics: {expected * 1e6:.4f} µm")

For comparison, the same bench with a sampled laser and numerical propagation. The symbolic profile needs
no sampling of the 4 mm beam at the lens, where the numerical bench needs a large grid.

In [ ]:
grid = focused.grid
numeric = Context()
laser = numeric.add(
    GaussianLaser(
        name="laser",
        wavelength=lam * u.m,
        waist=w0 * u.m,
        power=10 * u.mW,
        pixel_scale=40 * u.um,
        n_pixels=512,
    )
)
lens = numeric.add(ThinLens(name="lens", focal_length=f * u.m))
gap = numeric.add(FreeSpace(name="gap", distance=ctx["gap"].distance, output_grid=grid))
laser >> lens >> gap
sampled = numeric.run(["gap.out"])["gap.out"]

xx = np.asarray(grid.axes(backend="numpy")[0]) * 1e6
symbolic_profile = np.abs(np.asarray(to_numpy(to_wavefront(focused, grid).field)))[grid.ny // 2] ** 2
numeric_profile = np.abs(np.asarray(to_numpy(sampled.field)))[grid.ny // 2] ** 2
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(xx, symbolic_profile, label="closed form")
ax.plot(xx, numeric_profile, "--", label="numerical propagation")
ax.set(xlabel="x [µm]", ylabel="irradiance [W/m²]", title="Focused Gaussian beam")
ax.legend()
fig.tight_layout()
result.archive.save_figure(
    fig, "focused_beam", data={"x_um": xx, "closed_form": symbolic_profile, "numerical": numeric_profile}
)

## Tabulated transforms: the Airy pattern

A plane wave through a circular pupil, then a lens and a propagation over its focal length, gives the
Fraunhofer pattern of the pupil in closed form (Goodman, §5.2): a Bessel function $J_1$.

In [ ]:
airy_ctx = Context()
wave = airy_ctx.add(
    SymbolicSource.plane_wave(name="wave", amplitude=1 * u.W**0.5 / u.m, wavelength=500 * u.nm)
)
pupil = airy_ctx.add(CircularAperture(name="pupil", diameter=10 * u.mm))
lens = airy_ctx.add(ThinLens(name="lens", focal_length=1 * u.m))
focal = airy_ctx.add(FreeSpace(name="focal", distance=1 * u.m))
wave >> pupil >> lens >> focal
airy = airy_ctx.run(["focal.out"])["focal.out"]
sp.simplify(sp.Abs(airy.expression))

In [ ]:
psf_grid = Grid(256, dx=2 * u.um)
psf = np.abs(np.asarray(to_numpy(to_wavefront(airy, psf_grid).field))) ** 2
half = psf_grid.nx * 2 / 2  # µm
extent = [-half, half, -half, half]
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(np.log10(psf / psf.max() + 1e-8), extent=extent, origin="lower", cmap="inferno", vmin=-6)
ax.set(xlabel="x [µm]", ylabel="y [µm]", title="Airy pattern (log scale)")
fig.tight_layout()
result.archive.save_figure(fig, "airy", data={"psf": psf})

## Logged fallback

A pupil with spider vanes has no closed form: the field is sampled on its grid hint, the numerical transfer
runs, and the journal says so.

In [ ]:
fallback = Context()
beam = fallback.add(
    SymbolicSource.gaussian_beam(
        name="beam", waist=3 * u.mm, wavelength=633 * u.nm, power=1 * u.mW, grid=Grid(256, dx=40 * u.um)
    )
)
vanes = fallback.add(CircularAperture(name="pupil", diameter=8 * u.mm, n_spiders=4, spider_width=0.3 * u.mm))
beam >> vanes >> fallback.add(PowerMeter(name="meter"))
out = fallback.run(["pupil.out", "meter.out"])
print(type(out["pupil.out"]).__name__, out["meter.out"].data)
[entry.message for entry in fallback.journal.entries("log") if "pupil" in entry.message]

## From grids back to expressions

`lumos.symbolic.reconstruct` turns sampled fields into expressions, with the residual error of the
reconstruction. Here, the phase of an aberrated pupil is projected on Zernike modes (Noll 1976), and a
measured beam is fitted by a Gaussian with curvature.

In [ ]:
pupil_grid = Grid(128, dx=1 / 120 * u.m)
injected = np.array([0, 0, 0, 0.3, 0, 0.1, -0.2, 0, 0, 0.05])  # radians, Noll 1 to 10
basis = zernike_basis(pupil_grid, 1 * u.m, len(injected))
aberrated = Wavefront(
    (basis[0] != 0) * np.exp(1j * np.tensordot(injected, basis, axes=1)), pupil_grid, 1 * u.um
)
zernike = reconstruct.project_zernike(aberrated, diameter=1 * u.m, n_modes=10)
print("recovered:", np.round(zernike.coefficients, 6), "residual:", f"{zernike.residual:.1e}")

In [ ]:
measured = to_wavefront(
    gaussian_beam(waist=0.4 * u.mm, wavelength=633 * u.nm, power=1 * u.mW, distance_from_waist=0.3 * u.m),
    Grid(128, dx=10 * u.um),
)
fit = reconstruct.fit_gaussian(measured)
{key: value for key, value in fit.coefficients.items() if key in ("waist", "radius")}, fit.residual